# Deep Learning for Computer Vision

---

**Goethe University Frankfurt am Main**

Winter Semester 2025/26

<br>

## *Assignment 1 (Images)*

---

**Contact:** Matthias Fulde ([fulde@cs.uni-frankfurt.de](mailto:fulde@cs.uni-frankfurt.de))<br>

<br>

## Table of Contents

---

- [1 Loading Images](#1-Loading-Images)
- [2 Color Spaces](#2-Color-Spaces)
  - [2.1 Image Segmentation](#2.1-Image-Segmentation)
- [3 Geometric Transformations](#3-Geometric-Transformations)
  - [3.1 Scaling](#3.1-Scaling)
  - [3.2 Translation](#3.2-Translation)
  - [3.3 Rotation](#3.3-Rotation)


<br>

## Setup

---

In this notebook we use the libraries **Matplotlib**, **NumPy** and **OpenCV**. We recommend to use at least the `3.x` versions of OpenCV. You can check your version with the statements below.

In [ ]:
import numpy as np
import cv2 as cv

# Check installed OpenCV version.
cv.__version__

We want Matplotlib figures to appear within the notebook rather than inside a separate window, which is default in some environments, therefore we make use of the `%matplotlib` magic function to set the Matplotlib backend to `inline`. We also set the default color map in order to show grayscale images correctly and prevent Matplotlib from interpolating between pixels when images are scaled.

In [ ]:
import matplotlib.pyplot as plt

# Set default color map and interpolation method for images.
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['image.interpolation'] = 'nearest'

%matplotlib inline

<br>

## Definitions

---

The calls into the Matplotlib library necessary to `display` images quickly become repetetive, so we define a utility function for this task. The function expects as arguments a tuple with the width and height of the figure in which the images should appear, and a list of tuples, each containing the image data, the title and the position of an image.

The image position within the figure passed to the [subplot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplot.html) method can be given as a tuple or—for a smaller number of images—as a number consisting of three digits, which is interpreted as

$$
    \text{<position>} := \text{<number-of-rows><number-of-columns><index>}.
$$

<br>

In [ ]:
def display(figsize, images):
    """
    Display one or more images in a single figure.

    Parameters:
        - figsize: Size of the figure with (width, height).
        - images: List of tuples with (image, title, position).

    """
    plt.figure(figsize=figsize)

    for image, title, position in images:

        # Create titled subplot at given position.
        plt.subplot(*position if isinstance(position, tuple) else [position])
        plt.title(title)

        # Disable axis when only showing images.
        plt.axis('off')

        # Render image into current subplot.
        plt.imshow(image)

    # Display the figure.
    plt.show()

<br>

## Exercises

---

### 1 Loading Images

---

We use the OpenCV function [imread](https://docs.opencv.org/4.6.0/d4/da8/group__imgcodecs.html#ga288b8b3da0892bd651fce07b3bbd3a56) to read an image into a NumPy array. The first argument of the function is an absolute or relative path to the image and the second argument is a flag, which specifies the way the image should be read.

The following options to read images exist:

* `cv.IMREAD_COLOR`: Loads a color image. Any transparency (alpha channel) of the image will be stripped. *This is the default.*
* `cv.IMREAD_GRAYSCALE`: Loads an image in grayscale mode.
* `cv.IMREAD_UNCHANGED`: Loads a color image with an alpha channel, if included.

<br>

#### 1.1 Note

An important point to notice is that for historical reasons, a color image read with OpenCV is in Blue Green Red (BGR) mode, while Matplotlib displays images in the commonly used RGB mode. So color images will not be displayed correctly in Matplotlib if the image is read with OpenCV. We'll take care of this issue soon.

---

Let's see an example how loading images works.

In [ ]:
# Path to file.
filename = 'images/jupiter.png'

# Load images and render them into a figure.
display((15, 5), [
    (cv.imread(filename, cv.IMREAD_GRAYSCALE), 'Grayscale', 131),
    (cv.imread(filename, cv.IMREAD_COLOR),     'Color',     132),
    (cv.imread(filename, cv.IMREAD_UNCHANGED), 'Unchanged', 133)
])

<br>

#### 1.2 Question

How many channels are used for these images?

<br>

#### 1.3 Answer

* Grayscale:
* Color:
* Unchanged:

<!-- Hint: If not sure, you can just check the shape attribute of the images. -->

<br>

### 2 Color Spaces

---

To convert an image from one color space into another, we can use the OpenCV function [cvtColor](https://docs.opencv.org/4.6.0/d8/d01/group__imgproc__color__conversions.html#ga397ae87e1288a81d2363b61574eb8cab).

The function has two mandatory parameters, the source image whose color space should be changed and the color space conversion code. For a list of available codes follow the link above.

Converting between the BGR and RGB color spaces can be achieved using the `COLOR_BGR2RGB` code. The following example shows how to perform this conversion with images that have an alpha channel.

In [ ]:
# Load image with transparancy.
image = cv.imread('images/jupiter.png', cv.IMREAD_UNCHANGED)

# Switch red and blue channel.
result = cv.cvtColor(image, cv.COLOR_BGRA2RGBA)

# Show both images side by side.
display((10, 5), [
    (image,  'Original Image',  121),
    (result, 'Converted Image', 122)
])

<br>

### 2.1 Image Segmentation

---

Our goal in this exercise is to use the representation of an image in the HSV color space to partition a [radar image](https://public.nrao.edu/gallery/radar-image-of-mars/) of the planet Mars into segments of low, medium and high radar reflectivity.

Before we go into more detail on how to accomplish this, let's quickly recap the properties of the HSV color space.

The acronym HSV stands for *Hue*, *Saturation* and *Value*.

- **Hue** represents the color itself.<br>Given in degrees, the ranges for primary colors are

    -   0°-120° for *red*
    - 120°-240° for *green*
    - 240°-360° for *blue*

  and the ranges for secondary colors are

    -  60°-120° for *yellow*
    - 180°-240° for *cyan*
    - 300°-360° for *magenta*

    <br>

- **Saturation** represents the amount of gray in the color, where the minimum value is gray and the maximum value is the pure color.

- **Value** is the intensity of the color, which roughly corresponds to the color's lightness.

Let's have a look at the image we want to work with.

In [ ]:
# Load color image.
image = cv.imread('images/mars.png', cv.IMREAD_COLOR)

# Swap red and blue channel.
image = cv.cvtColor(image, cv.COLOR_BGR2RGB)

# Show transformed image.
display((8, 8), [
    (image, 'Radar Image of Mars', 111)
])

The red and yellow colors mark the regions with the highest reflectivity. In the regions colored blue, most of the radio waves are scattered into different directions or absorbed by the material. We consider the green areas as regions with a medium level of reflectivity.

<br>

#### 2.1.1 Task

Detect and extract from the image the regions of *low*, *medium* and *high* radar reflectivity on the planet. The result should be three images that only contain the respective regions of interest.

You can follow these steps:

1. Use the `cvtColor` method introduced above to convert the `image` from RGB to HSV, since this color space is easier to work with.
2. Find a lower and upper bound for the color that marks the region of interest by trying out a couple of values.
3. Use the method [inRange](https://docs.opencv.org/4.6.0/d2/de8/group__core__array.html#ga48af0ab51e36436c5d04340e036ce981) to create a mask from the lower and upper bound for the region.
4. Extract the region by applying [bitwise_and](https://docs.opencv.org/4.6.0/d2/de8/group__core__array.html#ga60b4d04b251ba5eb1392c34425497e14) to the original image using the mask.

Your result doesn't have to be pixel perfect. You only have to approximate the boundaries of the segments reasonably well.

<br>

#### 2.1.2 Hints

- You should check the actual ranges for hue, saturation and value in the OpenCV inplementation of HSV images.
- In order to separate areas of the planet with low reflectivity from the background noise, you can use the [medianBlur](https://docs.opencv.org/4.6.0/d4/d86/group__imgproc__filter.html#ga564869aa33e58769b4469101aac458f9) filter with a larger kernel size.


<br>

#### 2.1.3 Solution

Enter your solution in the code cell below. Store the segmented images in the given variables.

In [ ]:
############################################################
##                   START OF YOUR CODE                   ##
############################################################

low = image

medium = image

high = image

############################################################
##                    END OF YOUR CODE                    ##
############################################################

<br>

#### 2.1.4 Results

Use the following statements to display your results.

In [ ]:
# Display the extracted regions of interest together with the original radar image.
display((20, 5), [
    (image,  'Original Image',      141),
    (low,    'Low Reflectivity',    142),
    (medium, 'Medium Reflectivity', 143),
    (high,   'High Reflectivity',   144)
])

<br>

### 3 Geometric Transformations

---

### 3.1 Scaling

---

To scale an image we can use the OpenCV function [resize](https://docs.opencv.org/4.6.0/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231edc7e7529d).

The first parameter of the function is the source image and the second parameter is the size of the output image. Instead of giving the desired size of the output image directly, this mandatory parameter can be set to `None` and the named parameters `fx` and `fy` can be used to provide the scale factors for both axes.

Moreover, the named parameter `interpolation` allows to select an interpolation method.

---

In this exercise we want to resize the following image of the asteroid [Ida](https://photojournal.jpl.nasa.gov/catalog/PIA00135), taken by the Galileo orbiter:

In [ ]:
# Load and show grayscale image.
image = cv.imread('images/ida.png', cv.IMREAD_GRAYSCALE)

display((10, 6), [
    (image, 'Asteroid Ida', 111)
])

<br>

#### 3.1.1 Task

Resize the image of the asteroid such that the size of the larger side is exactly 1200 and the aspect ratio of the image is best preserved. Use bicubic interpolation for the scaling.

<br>

#### 3.1.2 Solution

Write your solution in the code cell below. Store the result in the given variable.

In [ ]:
############################################################
##                   START OF YOUR CODE                   ##
############################################################

result = image

############################################################
##                    END OF YOUR CODE                    ##
############################################################

<br>

#### 3.1.3 Test

You can check your code with the following statements.

In [ ]:
# Test size and aspect ratio of the resized image.
(H, W), (h, w) = image.shape, result.shape

difference = abs(W/H - w/h)

print(w == 1200 and difference < 1e-3)

<br>

### 3.2 Translation

---

Translation is the shifting of an object's location. Supposed we know the shift in $(x,y)$ direction, let it be $(t_x,t_y)$, then we can create a transformation matrix $M$ as follows:

$$
    M =
    \begin{bmatrix}
        1 & 0 & t_x \\
        0 & 1 & t_y
    \end{bmatrix}
$$

<br>

We can translate an image by passing the image and the transformation matrix $M$ as a NumPy array to the OpenCV function [warpAffine](https://docs.opencv.org/4.6.0/da/d54/group__imgproc__transform.html#ga0203d9ee5fcd28d40dbc4a1ea4451983), which generally can be used to apply affine transformations to an image.

This function has three mandatory parameters. The first one is the source image, the second is the transformation matrix and the third is the desired size of the output image. Check the linked documentation of the function to see what other options are available.

<br>

#### 3.2.1 Note

The data type of the array representing $M$ must be `float32` or `float64`.

---

In this exercise we want to write a generic function that allows us to center asteroids within an image. As example inputs to this function we use images of the asteroids [Itokawa](https://solarsystem.nasa.gov/asteroids-comets-and-meteors/asteroids/25143-itokawa/in-depth/) and [Vesta](https://solarsystem.nasa.gov/asteroids-comets-and-meteors/asteroids/4-vesta/in-depth/), where the objects of interest are placed on the right hand side and on the left hand side of the image, respectively.

Let's see how these images look like.

In [ ]:
# Read images in grayscale mode.
image1 = cv.imread('images/itokawa.png', cv.IMREAD_GRAYSCALE)
image2 = cv.imread('images/vesta.png',   cv.IMREAD_GRAYSCALE)

# Show them side by side.
display((16, 5), [
    (image1, 'Asteroid Itokawa', 121),
    (image2, 'Asteroid Vesta',   122)
])

<br>

#### 3.2.2 Task

Complete the definition of the `center` function below, so that takes an grayscale image of an asteroid like the ones shown above and returns an image of the same size, where the asteroid is horizontally and vertically centered.

In order to accomplish this, you can use the [moments](https://docs.opencv.org/4.6.0/d3/dc0/group__imgproc__shape.html#ga556a180f43cab22649c23ada36a8a139) function to compute the mean of the pixel distribution in the image and use this to translate the image accordingly. If the result of calling the function is stored in the variable `m`, the coordinates are given by

```Python
cx = round(m['m10'] / m['m00'])
cy = round(m['m01'] / m['m00'])
```

Your solution doesn't have to be pixel perfect. It's sufficient if the asteroids are more or less in the middle of the image.

<br>

#### 3.2.3 Solution

Enter your solution in the code cell below. Store the results in the given variable.

In [ ]:
def center(image):
    """
    Takes a grayscale image of an asteroid and translates the object to the center.
    The image returned has the same size as the input image.

    Parameters:
        - image: The input image.

    Returns:
        - out: The image with the object centered.

    """
    ############################################################
    ##                   START OF YOUR CODE                   ##
    ############################################################

    out = image

    ############################################################
    ##                    END OF YOUR CODE                    ##
    ############################################################
    return out

<br>

#### 3.2.4 Results

Use the following statement to show your results.

In [ ]:
# Show translated images.
display((16, 5), [
    (center(image1), 'Asteroid Itokawa', 121),
    (center(image2), 'Asteroid Vesta',   122)
])

<br>

### 3.3 Rotation

---

The `warpAffine` function described in the previous section can be used for rotations, too. We don't have to create the transformation matrix by hand, though. Instead, we can use the OpenCV function [getRotationMatrix2D](https://docs.opencv.org/4.6.0/da/d54/group__imgproc__transform.html#gafbbc470ce83812914a70abfb604f4326) to calculate an affine matrix for 2D rotation.

The function has three parameters:

1. The *center* of the rotation in the image.
2. The *rotation angle* in degrees, where positive values mean counter-clockwise rotation.
3. The *scale factor*, that is uniformly applied to both dimensions of the image.

---

In this exercise we want to rotate the following image of [Saturn](https://solarsystem.nasa.gov/resources/663/saturn-taken-from-voyager-2/?category=planets_saturn), taken by the Voyager 2 space probe:

In [ ]:
# Load color image.
image = cv.imread('images/saturn.jpg')

# Swap red and blue channel.
image = cv.cvtColor(image, cv.COLOR_BGR2RGB)

# Show the image.
display((5, 8), [
    (image, 'Saturn', 111)
])

<br>

#### 3.3.1 Task

Use the methods introduced above to rotate the image of Saturn such that the equator of the planet is roughly aligned with the horizontal axis of the image and no part of the image is cropped.

<br>

#### 3.3.2 Hints

You might notice that simply giving the image center and a suitable angle to the `getRotationMatrix2D` function and then pass the matrix together with the image to `warpAffine` doesn't do the trick.

If you follow the link above, you'll see that the rotation matrix is defined like this:

$$
    R =
    \begin{bmatrix}
        \alpha & \beta  & (1-\alpha) \cdot c_x - \beta \cdot c_y \\
        -\beta & \alpha & \beta \cdot c_x - (1-\alpha) \cdot c_y
    \end{bmatrix}
$$

where $(c_x, c_y)$ is the center of the image and

\begin{align*}
    \alpha & = s \cdot \cos(\theta) \\
    \beta  & = s \cdot \sin(\theta) 
\end{align*}

with scaling factor $s$ and angle $\theta$.

In order to rotate the image without cropping, you can extract the cosine and sine from the rotation matrix and compute the size of the output image with

\begin{align*}
    w_\text{out} & = h_\text{in} \cdot \beta  + w_\text{in} \cdot \alpha \\
    h_\text{out} & = h_\text{in} \cdot \alpha + w_\text{in} \cdot \beta.
\end{align*}

<br>

From $(w_\text{out}, h_\text{out})$ you can compute the new center and add the difference to $(c_x, c_y)$ to the translation part of the affine matrix, before you apply it to the source image.

<br>

#### 3.3.3 Solution

Write your code in the code cell below. Store the result in the given variable.

In [ ]:
############################################################
##                   START OF YOUR CODE                   ##
############################################################

result = image

############################################################
##                    END OF YOUR CODE                    ##
############################################################

<br>

#### 3.3.4 Result

Display your result with the following statement.

In [ ]:
# Show rotated image.
display((8, 6), [
    (result, 'Saturn Rotated', 111)
])